In [1]:
import zipfile
import os
import splitfolders
from tensorflow import keras
import numpy as np
from tqdm import tqdm
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model

In [ ]:
zip_path = "/content/drive/MyDrive/Breed Classification/bovine_breeds.zip"
extract_dir = "/content/drive/MyDrive/Breed Classification/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
data_dir = '/content/drive/MyDrive/Breed Classification/BovineBreeds_Augmented'
for cls in os.listdir(data_dir):
    path = os.path.join(data_dir, cls)
    print(f"{cls}: {len(os.listdir(path))} images")

In [ ]:
input_folder = "/content/drive/MyDrive/Breed Classification/BovineBreeds_Augmented"
output_folder = "/content/drive/MyDrive/Breed Classification/split_data"

splitfolders.ratio(input_folder,
                   output=output_folder,
                   seed=42,
                   ratio=(0.7, 0.2, 0.1))

In [2]:
train_dir = '/content/drive/MyDrive/Breed Classification/split_data/train'

class_names_from_dir = sorted(os.listdir(train_dir))
print(class_names_from_dir)

['.ipynb_checkpoints', 'Alambadi', 'Amritmahal', 'Ayrshire', 'Banni', 'Bargur', 'Bhadawari', 'Brown_Swiss', 'Dangi', 'Deoni', 'Gir', 'Holstein_Friesian', 'Jaffrabadi', 'Jersey', 'Kangayam', 'Kasargod', 'Kenkatha', 'Kherigarh', 'Krishna_Valley', 'Mehsana', 'Murrah', 'Nagori', 'Nili_Ravi', 'Nimari', 'Ongole', 'Red_Sindhi', 'Sahiwal', 'Surti', 'Tharparkar', 'Umblachery']


In [ ]:
IMG_SIZE = (300, 300)


train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,  # ✅ EfficientNet-specific
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
)


val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
)

train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/Breed Classification/split_data/train',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'
)


val_generator = val_datagen.flow_from_directory(
    '/content/drive/MyDrive/Breed Classification/split_data/val',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical',

)



test_generator = val_datagen.flow_from_directory(
    '/content/drive/MyDrive/Breed Classification/split_data/test',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'

)

Found 5062 images belonging to 30 classes.
Found 1439 images belonging to 30 classes.
Found 749 images belonging to 30 classes.


In [ ]:
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x1 = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
x2 = layers.Dropout(0.2)(x1)
x3 = Dense(64, activation = 'relu', kernel_regularizer= regularizers.l2(0.001))(x2)
x4 = layers.Dropout(0.2)(x3)
x5 = Dense(30, activation='softmax')(x4)
model = Model(inputs=base_model.input, outputs=x5)


model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/Breed Classification/model/best_model.keras",
    monitor="val_accuracy",        # metric to monitor
    mode="max",                    # because we want maximum accuracy
    save_best_only=True,           # saves only when model improves
    verbose=1
)

early_stop = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)



history = model.fit(train_generator, validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint],
    verbose = 1

)

In [ ]:
model_path = "/content/drive/MyDrive/Breed Classification/model/best_model.keras"

loaded_model = load_model(model_path)

loss, accuracy = loaded_model.evaluate(test_generator)
print(loss)
print(accuracy)

Copying files: 3 files [18:37, 372.45s/ files]
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


24/24 ━━━━━━━━━━━━━━━━━━━━ 280s 11s/step - accuracy: 0.7935 - loss: 1.0418
1.1101150512695312
0.7810413837432861
